---
## Latihan Mandiri

Buat SparkSession baru terlebih dahulu (copy cell dari Sub-bab 4.5) sebelum mengerjakan latihan berikut.

**SPARK SESSION**

In [128]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

# Membuat SparkSession — "local[*]" berarti gunakan seluruh core CPU yang tersedia di VM
spark = SparkSession.builder \
    .appName("Pertemuan4-PengenalanPySpark") \
    .master("local[*]") \
    .getOrCreate()

# Mengurangi banyaknya pesan log teknis agar output lebih bersih
spark.sparkContext.setLogLevel("ERROR")

print("SparkSession berhasil dibuat!")
print("Versi Spark:", spark.version)

SparkSession berhasil dibuat!
Versi Spark: 3.5.9


**Memuat Data ke Spark DataFrame**

In [129]:
# Membuat ulang dataset contoh (identik dengan Tugas Mandiri Pertemuan 2) jika belum ada
import os
if not os.path.exists("data_transaksi_ecommerce.csv"):
    import numpy as np
    import pandas as pd
    np.random.seed(42)
    n = 600
    kategori_list = ["Elektronik", "Fashion", "Makanan & Minuman", "Kesehatan & Kecantikan", "Rumah Tangga"]
    kota_list = ["Magelang", "Yogyakarta", "Semarang", "Solo", "Purworejo"]
    metode_bayar_list = ["Transfer Bank", "E-Wallet", "COD", "Kartu Kredit"]
    tanggal_range = pd.date_range("2026-07-01", "2026-07-31", freq="D")
    data = {
        "order_id": [f"ORD-{1000+i}" for i in range(n)],
        "tanggal": np.random.choice(tanggal_range, size=n).astype(str),
        "kategori": np.random.choice(kategori_list, size=n, p=[0.25,0.25,0.20,0.15,0.15]),
        "kota": np.random.choice(kota_list, size=n),
        "unit_terjual": np.random.randint(1, 10, size=n),
        "harga_satuan": np.random.choice([25000,50000,75000,100000,150000,250000,500000], size=n),
        "metode_pembayaran": np.random.choice(metode_bayar_list, size=n, p=[0.35,0.30,0.20,0.15]),
    }
    pd.DataFrame(data).to_csv("data_transaksi_ecommerce.csv", index=False)
    print("Dataset dibuat ulang.")
else:
    print("Dataset sudah tersedia.")

Dataset sudah tersedia.


In [130]:
# Jalankan ini dulu sebelum mengerjakan latihan di bawah
spark = SparkSession.builder.appName("Latihan4").master("local[*]").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
df = spark.read.csv("data_transaksi_ecommerce.csv", header=True, inferSchema=True)
df = df.withColumn("total_pendapatan", col("unit_terjual") * col("harga_satuan"))
print("Siap. Jumlah baris:", df.count())

Siap. Jumlah baris: 600


**Soal 1.** Tampilkan hanya kolom `order_id`, `kategori`, dan `total_pendapatan` untuk transaksi dengan `metode_pembayaran` bernilai `"E-Wallet"`.

In [131]:
df.filter(col("metode_pembayaran") == "E-Wallet").select("order_id", "kategori", "total_pendapatan").show()

+--------+--------------------+----------------+
|order_id|            kategori|total_pendapatan|
+--------+--------------------+----------------+
|ORD-1000|Kesehatan & Kecan...|         2250000|
|ORD-1003|          Elektronik|          150000|
|ORD-1006|   Makanan & Minuman|           25000|
|ORD-1013|   Makanan & Minuman|          900000|
|ORD-1014|             Fashion|           75000|
|ORD-1015|          Elektronik|          375000|
|ORD-1017|Kesehatan & Kecan...|         4000000|
|ORD-1019|          Elektronik|          675000|
|ORD-1021|          Elektronik|          200000|
|ORD-1022|        Rumah Tangga|          900000|
|ORD-1023|Kesehatan & Kecan...|         2500000|
|ORD-1024|             Fashion|          750000|
|ORD-1027|             Fashion|           50000|
|ORD-1029|             Fashion|          600000|
|ORD-1030|             Fashion|          300000|
|ORD-1033|          Elektronik|          300000|
|ORD-1035|   Makanan & Minuman|         2500000|
|ORD-1036|   Makanan

**Soal 2.** Hitung total pendapatan **per kategori** (bukan per kota), urutkan dari yang tertinggi.

In [132]:
df.groupBy("kategori").sum("total_pendapatan").withColumnRenamed("sum(total_pendapatan)", "total_pendapatan").orderBy(col("total_pendapatan").desc()).show()

+--------------------+----------------+
|            kategori|total_pendapatan|
+--------------------+----------------+
|             Fashion|       124825000|
|          Elektronik|       110600000|
|   Makanan & Minuman|        95400000|
|Kesehatan & Kecan...|        82200000|
|        Rumah Tangga|        77350000|
+--------------------+----------------+



**Soal 3.** Tampilkan jumlah transaksi untuk masing-masing `metode_pembayaran` (tidak pakai `agg`).

In [133]:
df.groupBy("metode_pembayaran").count().show()

+-----------------+-----+
|metode_pembayaran|count|
+-----------------+-----+
|              COD|  114|
|    Transfer Bank|  208|
|     Kartu Kredit|   85|
|         E-Wallet|  193|
+-----------------+-----+



**Soal 4 (Refleksi singkat).** Dalam 2-3 kalimat: apa yang dimaksud dengan *lazy evaluation* di PySpark, dan mengapa hal ini menguntungkan ketika bekerja dengan data berskala besar? Tulis jawaban pada markdown cell di bawah ini.

Lazy evaluation adalah cara kerja PySpark yang menunda eksekusi perintah sampai hasilnya benar-benar dibutuhkan. Hal ini menguntungkan karena PySpark dapat mengoptimalkan proses dan mengurangi pekerjaan yang tidak perlu saat mengolah data dalam jumlah besar.


---
## TUGAS MANDIRI (Dikerjakan Selama 1 Minggu)

>  **Tenggat waktu:** dikumpulkan paling lambat **sebelum Pertemuan 5 dimulai**.
>  **Sifat tugas:** individu.

### Konteks / Skenario

Tim engineering platform e-commerce (skenario yang sama dari Pertemuan 2-3) resmi meminta seluruh proses analisis data yang sebelumnya memakai pandas **dipindahkan ke PySpark**, karena volume data transaksi diperkirakan akan tumbuh sangat besar dalam waktu dekat sehingga pandas (yang memuat semua data ke RAM) tidak lagi memadai. Sebagai data analyst yang baru belajar PySpark, anda ditugaskan membuktikan bahwa seluruh alur analisis dapat direplikasi menggunakan PySpark, **membaca data langsung dari HDFS**.

### Menyiapkan Dataset

Jalankan cell berikut untuk membuat dataset baru (transaksi bulan September 2026, lebih banyak baris dari sebelumnya) dan mengunggahnya ke HDFS.

In [134]:
# Sel ini membuat dataset baru untuk Tugas Mandiri Pertemuan 4 dan mengunggahnya ke HDFS
import numpy as np
import pandas as pd

np.random.seed(99)
n = 1000
kategori_list = ["Elektronik", "Fashion", "Makanan & Minuman", "Kesehatan & Kecantikan", "Rumah Tangga", "Olahraga"]
kota_list = ["Magelang", "Yogyakarta", "Semarang", "Solo", "Purworejo", "Kebumen"]
metode_bayar_list = ["Transfer Bank", "E-Wallet", "COD", "Kartu Kredit"]
tanggal_range = pd.date_range("2026-09-01", "2026-09-30", freq="D")

data = {
    "order_id": [f"ORD-{3000 + i}" for i in range(n)],
    "tanggal": np.random.choice(tanggal_range, size=n).astype(str),
    "kategori": np.random.choice(kategori_list, size=n),
    "kota": np.random.choice(kota_list, size=n),
    "unit_terjual": np.random.randint(1, 12, size=n),
    "harga_satuan": np.random.choice([20000, 45000, 60000, 90000, 125000, 200000, 350000], size=n),
    "metode_pembayaran": np.random.choice(metode_bayar_list, size=n),
    "rating": np.random.choice([1, 2, 3, 4, 5, np.nan], size=n, p=[0.03, 0.02, 0.10, 0.30, 0.35, 0.20]),
}
df_tugas4 = pd.DataFrame(data)
df_tugas4.to_csv("transaksi_september_2026.csv", index=False)
print(f"Dataset dibuat: {df_tugas4.shape[0]} baris")

# Mengunggah ke HDFS
!hdfs dfs -mkdir -p /user/mahasiswa/tugas4
!hdfs dfs -put -f transaksi_september_2026.csv /user/mahasiswa/tugas4/
print("Berhasil diunggah ke HDFS: /user/mahasiswa/tugas4/transaksi_september_2026.csv")

Dataset dibuat: 1000 baris
Berhasil diunggah ke HDFS: /user/mahasiswa/tugas4/transaksi_september_2026.csv


### Instruksi Pengerjaan

Buat notebook baru **`Tugas4_[NPM]_[Nama Lengkap].ipynb`**, buat `SparkSession` baru, lalu kerjakan bagian **A sampai E** berikut — **seluruhnya wajib menggunakan PySpark, bukan pandas**, dan data **wajib dibaca langsung dari HDFS** (`hdfs://localhost:9000/...`), bukan dari berkas lokal.

---

**A. Membaca dan Eksplorasi Awal** *(bobot 15%)*

Baca dataset dari HDFS, tampilkan `printSchema()`, jumlah baris (`count()`), dan 10 baris pertama (`show(10)`).

**B. Menangani Data Kosong** *(bobot 15%)*

Kolom `rating` memiliki nilai kosong. Tampilkan berapa banyak, lalu gunakan `df.na.fill()` atau `df.na.drop()` (pilih salah satu, jelaskan alasannya pada markdown cell) untuk menanganinya.

**C. Transformasi Data** *(bobot 20%)*

Tambahkan kolom `total_pendapatan` (`unit_terjual x harga_satuan`), lalu tambahkan kolom `tier_transaksi` yang bernilai `"Besar"` jika `total_pendapatan > 500000`, atau `"Kecil"` jika sebaliknya 

**D. Analisis dengan GroupBy** *(bobot 30%)*

Jawablah dengan kode PySpark (bukan pandas):
1. Kategori apa yang memiliki `total_pendapatan` tertinggi?
2. Kota mana dengan jumlah transaksi **tier "Besar"** terbanyak?
3. Berapa rata-rata `rating` untuk masing-masing `metode_pembayaran` (data kosong sudah ditangani di bagian B)?

**E. Menyimpan Hasil ke HDFS** *(bobot 20%)*

Simpan DataFrame hasil olahan bagian C (lengkap dengan kolom `total_pendapatan` dan `tier_transaksi`) ke HDFS dalam format CSV baru, kemudian verifikasi apakah sudah berhasil.

> **Catatan:** Spark menyimpan hasil sebagai **beberapa berkas partisi** (`part-00000...`, dst.), bukan satu berkas tunggal seperti pandas — ini normal dan justru mencerminkan sifat terdistribusi Spark. Jelaskan secara singkat pada markdown cell mengapa hal ini terjadi

---

### Ketentuan Pengumpulan

- Kumpulkan `Tugas4_[NIM]_[Nama Lengkap].ipynb` melalui ELITA, paling lambat **1 minggu dari hari ini, pukul 23.59 WIB**.
- Pastikan Hadoop aktif dan seluruh cell sudah dijalankan (**Run All**) sebelum dikumpulkan.
- **Dilarang menggunakan pandas** untuk bagian analisis A-E (boleh dipakai hanya di sel pembuatan dataset yang sudah disediakan).

### Rubrik Penilaian

| Bagian | Kriteria | Bobot |
|---|---|---|
| A. Baca & Eksplorasi | Data berhasil dibaca dari HDFS, struktur & jumlah baris benar | 15% |
| B. Data Kosong | Missing value teridentifikasi & ditangani dengan alasan yang logis | 15% |
| C. Transformasi | Kedua kolom baru dihitung dengan benar menggunakan fungsi PySpark yang tepat | 20% |
| D. Analisis GroupBy | Ketiga jawaban analitis benar & menggunakan sintaks PySpark (bukan pandas) | 30% |
| E. Simpan ke HDFS | Hasil berhasil tersimpan ke HDFS; penjelasan tentang partisi tepat | 20% |


In [135]:
#A. 

#Membaca dataset
df = spark.read.csv("transaksi_september_2026.csv", header=True, inferSchema=True)

#Menampilkan printSchema
print("Menampilkan printSchema")
df.printSchema()

#Menampilkan 10 baris pertama 
print("\nMenampilkan 10 baris pertama")
df.show(10)

#Menghitung jumlah baris 
print("\nMenampilkan jumlah baris")
print("Jumlah baris:", df.count())

Menampilkan printSchema
root
 |-- order_id: string (nullable = true)
 |-- tanggal: timestamp (nullable = true)
 |-- kategori: string (nullable = true)
 |-- kota: string (nullable = true)
 |-- unit_terjual: integer (nullable = true)
 |-- harga_satuan: integer (nullable = true)
 |-- metode_pembayaran: string (nullable = true)
 |-- rating: double (nullable = true)


Menampilkan 10 baris pertama
+--------+-------------------+--------------------+----------+------------+------------+-----------------+------+
|order_id|            tanggal|            kategori|      kota|unit_terjual|harga_satuan|metode_pembayaran|rating|
+--------+-------------------+--------------------+----------+------------+------------+-----------------+------+
|ORD-3000|2026-09-02 00:00:00|        Rumah Tangga|Yogyakarta|           3|       90000|              COD|   4.0|
|ORD-3001|2026-09-04 00:00:00|   Makanan & Minuman|      Solo|           3|      200000|         E-Wallet|   5.0|
|ORD-3002|2026-09-26 00:00:00|Keseh

In [136]:
#B

#Menghitung jumlah data kosong pada kolom rating
print("Jumlah rating kosong:", df.filter(col("rating").isNull()).count())

Jumlah rating kosong: 204


In [137]:
from pyspark.sql.functions import col, avg

#Menghitung rata-rata rating
rata_rating = df.select(avg("rating")).first()[0]

#Mengisi nilai rating yang kosong dengan rata-rata rating
df = df.na.fill({"rating": rata_rating})

#Menghitung kembali jumlah rating kosong setelah ditangani
print("Jumlah rating kosong setelah ditangani:", df.filter(col("rating").isNull()).count())

Jumlah rating kosong setelah ditangani: 0


Saya menggunakan df.na.fill() karena nilai rating yang kosong masih dapat diganti dengan nilai rata-rata rating. Dengan cara ini, data yang memiliki rating kosong tidak perlu dihapus sehingga jumlah data tetap sama.

In [138]:
#C

from pyspark.sql.functions import when

#Menambahkan kolom total_pendapatan dari unit_terjual dikali harga_satuan
df = df.withColumn("total_pendapatan", col("unit_terjual") * col("harga_satuan"))

#Menambahkan kolom tier_transaksi berdasarkan total_pendapatan
df = df.withColumn("tier_transaksi", when(col("total_pendapatan") > 500000, "Besar").otherwise("Kecil"))

#Menampilkan hasil transformasi
df.show()

+--------+-------------------+--------------------+----------+------------+------------+-----------------+------------------+----------------+--------------+
|order_id|            tanggal|            kategori|      kota|unit_terjual|harga_satuan|metode_pembayaran|            rating|total_pendapatan|tier_transaksi|
+--------+-------------------+--------------------+----------+------------+------------+-----------------+------------------+----------------+--------------+
|ORD-3000|2026-09-02 00:00:00|        Rumah Tangga|Yogyakarta|           3|       90000|              COD|               4.0|          270000|         Kecil|
|ORD-3001|2026-09-04 00:00:00|   Makanan & Minuman|      Solo|           3|      200000|         E-Wallet|               5.0|          600000|         Besar|
|ORD-3002|2026-09-26 00:00:00|Kesehatan & Kecan...|  Semarang|           8|       60000|         E-Wallet|               3.0|          480000|         Kecil|
|ORD-3003|2026-09-09 00:00:00|   Makanan & Minuman| 

In [139]:
#D

#Menjumlahkan total pendapatan berdasarkan kategori dan mengurutkan dari tertinggi
df.groupBy("kategori").sum("total_pendapatan").orderBy("sum(total_pendapatan)", ascending=False).show(1)

+------------+---------------------+
|    kategori|sum(total_pendapatan)|
+------------+---------------------+
|Rumah Tangga|            138665000|
+------------+---------------------+
only showing top 1 row



In [140]:
#Menghitung jumlah transaksi tier Besar berdasarkan kota dan mengurutkan dari terbanyak
df.filter(col("tier_transaksi") == "Besar").groupBy("kota").count().orderBy("count", ascending=False).show(1)

+----+-----+
|kota|count|
+----+-----+
|Solo|   92|
+----+-----+
only showing top 1 row



In [141]:
#Menghitung rata-rata rating untuk setiap metode pembayaran
df.groupBy("metode_pembayaran").avg("rating").show()

+-----------------+------------------+
|metode_pembayaran|       avg(rating)|
+-----------------+------------------+
|              COD| 4.167310656870009|
|    Transfer Bank|   4.1592349097265|
|     Kartu Kredit|4.1179474608816475|
|         E-Wallet| 4.137728643216084|
+-----------------+------------------+



In [142]:
#E

#Menyimpan DataFrame hasil olahan ke HDFS dalam format CSV
df.write.mode("overwrite").option("header", True).csv("hdfs://localhost:9000/user/mahasiswa/tugas4/hasil_transaksi_september_2026")

#Memastikan file berhasil disimpan
!hdfs dfs -ls /user/mahasiswa/tugas4/hasil_transaksi_september_2026

Found 2 items
-rw-r--r--   3 dian-khaerunia-azzahra supergroup          0 2026-09-12 21:35 /user/mahasiswa/tugas4/hasil_transaksi_september_2026/_SUCCESS
-rw-r--r--   3 dian-khaerunia-azzahra supergroup     100356 2026-09-12 21:35 /user/mahasiswa/tugas4/hasil_transaksi_september_2026/part-00000-c665ba9d-2161-4e9d-8181-a8ab26ce22c4-c000.csv


In [143]:
spark.stop()
print("SparkSession ditutup.")

SparkSession ditutup.
